# Next Activity Prediction com RAG: 1 log × 1 LLM local (Llama)

Este notebook roda o mesmo fluxo do `src/eval.py`, mas **para um único event log e um único modelo local (Llama via HuggingFace)**, pensado para um **runtime local do Colab** (Jupyter na sua máquina).

Fluxo: 1. Configuração → 2. Ambiente → 3. Imports → 4. Embeddings → 5. Vector store (Qdrant) → 6. Pré-processamento do log + indexação → 7. Test set → 8. Carregar o Llama (4-bit) → 9. Avaliação → 10. Resultados

Requisitos:
- GPU NVIDIA com CUDA (o modelo é carregado em 4-bit com `bitsandbytes`; o Llama-3.1-8B precisa de ~6–7 GB de VRAM).
- Token do HuggingFace (`HF_TOKEN`) nos **Secrets do Colab** (ícone de chave), no `.env` ou digitado quando pedido, com acesso liberado ao modelo Llama escolhido.
- Logs descompactados em `ensemble_rag_next_activity/logs/`.

## 1. Configuração
Tudo o que você precisa ajustar fica nesta célula.

In [ ]:
# ----------------------------- CONFIGURAÇÃO -----------------------------
# Caminho do repositório. Deixe None para detectar automaticamente
# (procura 'ensemble_rag_next_activity' a partir do diretório atual e dos diretórios pai).
REPO_DIR = None   # ex.: r"C:\Users\voce\projetos\ensemble_rag_next_activity" ou "/home/voce/ensemble_rag_next_activity"

# Event log (arquivo dentro de logs/)
LOG_NAME = "sepsis.xes"
# Opções: sepsis.xes, hospital_billing.xes, BPIC_2012.xes, BPIC_2013_closed_problems.xes,
#         BPIC_2013_incidents.xes, bpic20_international_declarations.xes, udonya.xes,
#         synthetic-13-multivariant.xes, synthetic-2-2var-1rel-1-nonrel.xes

# LLM local (família Llama suportada pelo pipeline.py)
LLM_ID = "meta-llama/Llama-3.1-8B-Instruct"
# Alternativas: "meta-llama/Llama-3.2-3B-Instruct", "meta-llama/Llama-3.2-1B-Instruct",
#               "meta-llama/Meta-Llama-3-8B-Instruct"

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L12-v2"
EVALUATION_MODALITY = "evaluation-attributes"
USE_RAG = True
NUM_DOCUMENTS_IN_CONTEXT = 3
MAX_NEW_TOKENS = 2048
MODEL_MAX_LENGTH = 128000      # apenas registrado no arquivo de resultados (igual ao eval.py)
BATCH_SIZE = 32
EARLYNESS_BUCKETS = [5, 10, 20, 30]
SEED = 10

# Vector store:
#   "memory" -> Qdrant em memória (não precisa de Docker)  [recomendado]
#   "server" -> Qdrant em Docker usando QDRANT_URL / QDRANT_GRPC_PORT do .env
QDRANT_MODE = "memory"
COLLECTION_NAME = "nap-rag"

# Test set:
#   True  -> usa o CSV já versionado em tests/test_sets/ (o mesmo do paper)
#   False -> gera um novo test set (30% das traces, até 300 prefixos), como o eval.py com --rebuild_db_and_tests True.
#            O CSV novo é salvo com sufixo "_regenerated" para não sobrescrever o original.
USE_EXISTING_TEST_SET = True

# Limite de casos de teste (None = todos, ~300). Use algo como 5 para um teste rápido.
MAX_TEST_CASES = None

# True imprime o prompt/resposta completos de cada predição (muito verboso)
VERBOSE = False

# Instalar as dependências do requirements.txt (deixe True só na primeira execução)
INSTALL_REQUIREMENTS = False

## 2. Ambiente: localizar o repo, instalar dependências, carregar `.env`

In [ ]:
import os, sys
from pathlib import Path

def find_repo(start: Path):
    for base in [start, *start.parents]:
        for cand in (base, base / "ensemble_rag_next_activity"):
            if (cand / "src" / "eval.py").exists() and (cand / "src" / "pipeline.py").exists():
                return cand.resolve()
    return None

REPO = Path(REPO_DIR).expanduser().resolve() if REPO_DIR else find_repo(Path.cwd())
assert REPO and (REPO / "src" / "pipeline.py").exists(), (
    "Repositório não encontrado. Defina REPO_DIR na célula de configuração.")
SRC = REPO / "src"
print("Repo:", REPO)

if INSTALL_REQUIREMENTS:
    %pip install -q -r {str(REPO / "requirements.txt")}

In [ ]:
from dotenv import load_dotenv

load_dotenv(REPO / ".env", override=True)   # o .env não vem no git clone; se não existir, segue para as outras fontes
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Token do HuggingFace: Colab Secrets -> .env / variável de ambiente -> digitar ---
HF_TOKEN = None
try:
    from google.colab import userdata          # Colab: ícone de chave (Secrets) -> nome HF_TOKEN
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
HF_TOKEN = (HF_TOKEN or os.getenv("HF_TOKEN") or "").strip()
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Cole seu token do HuggingFace (hf_...): ").strip()
assert HF_TOKEN.startswith("hf_"), "Token inválido/vazio. Gere um em https://huggingface.co/settings/tokens"
os.environ["HF_TOKEN"] = HF_TOKEN            # o pipeline.py lê daqui (os.getenv('HF_TOKEN'))

from huggingface_hub import login, whoami, model_info
login(token=HF_TOKEN, add_to_git_credential=False)
print("HuggingFace: logado como", whoami(token=HF_TOKEN)["name"])
try:
    model_info(LLM_ID, token=HF_TOKEN, files_metadata=False)
    from huggingface_hub import hf_hub_download
    hf_hub_download(LLM_ID, "config.json", token=HF_TOKEN)
    print("Acesso ao modelo OK:", LLM_ID)
except Exception as e:
    raise RuntimeError(
        f"Sem acesso a {LLM_ID}. Peça acesso em https://huggingface.co/{LLM_ID} (botão 'Agree and access') "
        f"e use um token com permissão de leitura de repositórios gated.\n{e}")

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

LOG_PATH = REPO / "logs" / LOG_NAME
assert LOG_PATH.exists(), f"Log não encontrado: {LOG_PATH}. Arquivos em logs/: {sorted(p.name for p in (REPO/'logs').glob('*'))}"
(REPO / "tests" / "validation").mkdir(parents=True, exist_ok=True)

import torch
assert torch.cuda.is_available(), "CUDA não disponível: o carregamento em 4-bit (bitsandbytes) do pipeline exige GPU NVIDIA."
DEVICE = f"cuda:{torch.cuda.current_device()}"

print("Log:", LOG_PATH, f"({LOG_PATH.stat().st_size/1e6:.1f} MB)")
print("GPU:", torch.cuda.get_device_name(0),
      f"| VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 3. Imports do projeto e seed

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import log_preprocessing as lp
import pipeline as p
import utility as u
import vector_store as vs
from oracle import VerificationOracle

u.seed_everything(SEED)

## 4. Modelo de embeddings

In [ ]:
embed_model, space_dimension = p.initialize_embedding_model(EMBED_MODEL_ID, DEVICE, BATCH_SIZE)
print("Dimensão dos embeddings:", space_dimension)

## 5. Vector store (Qdrant)
`memory` dispensa Docker; `server` usa o mesmo `vs.initialize_vector_store` do `eval.py`.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

if QDRANT_MODE == "memory":
    q_client = QdrantClient(location=":memory:")
    q_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=space_dimension, distance=Distance.COSINE),
    )
    q_store = QdrantVectorStore(q_client, collection_name=COLLECTION_NAME, embedding=embed_model)
elif QDRANT_MODE == "server":
    q_client, q_store = vs.initialize_vector_store(
        os.getenv("QDRANT_URL"), int(os.getenv("QDRANT_GRPC_PORT")),
        COLLECTION_NAME, embed_model, space_dimension, True)
else:
    raise ValueError("QDRANT_MODE deve ser 'memory' ou 'server'")
print("Vector store pronto:", QDRANT_MODE)

## 6. Pré-processamento do log e indexação dos prefixos
Igual ao `eval.py`: lê o XES, extrai traces, monta prefixos (base=1, gap=3) e grava no Qdrant. Para logs grandes (BPIC_2012, bpic20) isso pode demorar bastante.

In [ ]:
import contextlib, io

content = lp.read_event_log(LOG_NAME)
with contextlib.redirect_stdout(io.StringIO()):   # as funções imprimem uma linha por trace/prefixo
    traces = lp.extract_traces(content)
    prefixes = lp.build_prefixes(traces)
    prefixes, event_attributes, activities_set = lp.process_prefixes(prefixes, LOG_NAME)
del content
print(f"Traces (>=2 eventos): {len(traces)} | Prefixos únicos: {len(prefixes)} | Atividades: {len(activities_set)}")

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    n_stored = vs.store_traces(prefixes, q_client, LOG_NAME, embed_model, COLLECTION_NAME)
print("Pontos armazenados no Qdrant:", n_stored)

## 7. Test set

In [ ]:
base_name = f"test_set_{LOG_NAME.split('.xes')[0]}_{EVALUATION_MODALITY}"
test_sets_dir = REPO / "tests" / "test_sets"

if USE_EXISTING_TEST_SET:
    test_set_path = test_sets_dir / f"{base_name}.csv"
    assert test_set_path.exists(), f"Test set não encontrado: {test_set_path}. Use USE_EXISTING_TEST_SET = False."
    test_set_size = None
else:
    test_set_path = test_sets_dir / f"{base_name}_regenerated.csv"
    test_traces = lp.generate_test_set(traces, 0.3)
    test_set_size = len(test_traces)
    with contextlib.redirect_stdout(io.StringIO()):
        lp.generate_csv_from_test_set(test_traces, str(test_set_path), log_name=LOG_NAME)

test_list = u.load_csv_questions(str(test_set_path))   # caminho absoluto: os.path.join o mantém como está
if MAX_TEST_CASES:
    test_list = test_list[:MAX_TEST_CASES]
print("Test set:", test_set_path.name, "| casos:", len(test_list))
print("Exemplo:", test_list[0][0][:300], "...\n -> esperado:", test_list[0][1])

## 8. Carregar o Llama (4-bit, via `RAGPipeline`)

In [ ]:
pipeline_instance = p.RAGPipeline(LLM_ID, MAX_NEW_TOKENS, USE_RAG)
assert pipeline_instance.model_type == "local" and pipeline_instance.model_family == "metaai", (
    f"Modelo reconhecido como {pipeline_instance.model_family}/{pipeline_instance.model_type}; use um ID Llama listado em pipeline.py")
print(f"Modelo carregado: {LLM_ID} | VRAM alocada: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
run_data = {
    'Batch Size': BATCH_SIZE,
    'Embedding Model ID': EMBED_MODEL_ID,
    'Vector Space Dimension': space_dimension,
    'Evaluation Modality': EVALUATION_MODALITY,
    'Event Log': LOG_NAME,
    'Total Traces in Log': len(traces),
    'Test Set Size': test_set_size if test_set_size is not None else f"existing CSV ({test_set_path.name})",
    'Traces Stored Size': len(prefixes),
    'LLM ID': LLM_ID,
    'Context Window LLM': MODEL_MAX_LENGTH,
    'Max Generated Tokens LLM': MAX_NEW_TOKENS,
    'Number of Documents in the Context': NUM_DOCUMENTS_IN_CONTEXT,
    'Prefix Base': 1,   # valores efetivamente usados por lp.build_prefixes
    'Prefix Gap': 3,
    'Rebuilt Vector Index and Test Set': True,
    'RAG': USE_RAG,
    'Vector Store Mode': QDRANT_MODE,
    'Test Cases Evaluated': len(test_list),
    'Earlyness Boundaries': EARLYNESS_BUCKETS,
    'Event Attributes': str(event_attributes),
    'Activities': activities_set,
}

### Teste rápido com 1 prefixo
Confere o prompt e o formato da resposta antes de rodar tudo.

In [ ]:
prompt, answer = pipeline_instance.produce_answer(test_list[0][0], q_store, NUM_DOCUMENTS_IN_CONTEXT, run_data)
print("PROMPT (final):\n", prompt[-1500:])
print("\nRESPOSTA:\n", answer)
print("\nESPERADO:", test_list[0][1])

## 9. Avaliação

In [ ]:
from tqdm.auto import tqdm

oracle = VerificationOracle(run_data, EARLYNESS_BUCKETS)
hits = 0
pbar = tqdm(test_list, desc="Predições")
for i, (prefix, expected) in enumerate(pbar, 1):
    oracle.add_prefix_with_expected_answer_pair(prefix, expected)
    prompt, answer = pipeline_instance.produce_answer(prefix, q_store, NUM_DOCUMENTS_IN_CONTEXT, run_data)
    if VERBOSE:
        ok = oracle.verify_answer(prompt, prefix, answer)
    else:
        with contextlib.redirect_stdout(io.StringIO()):
            ok = oracle.verify_answer(prompt, prefix, answer)
    hits += bool(ok)
    pbar.set_postfix(acc=f"{hits/i:.3f}")

oracle.write_results_to_file()   # também chama compute_stats()
result_file = max((REPO / "tests" / "validation").glob("*.txt"), key=os.path.getmtime)
print("Resultados salvos em:", result_file)

## 10. Resultados

In [ ]:
import pandas as pd

summary = oracle.get_earlyness_summary()
overall = summary["overall_metrics"]
print(f"{LLM_ID} | {LOG_NAME} | RAG={USE_RAG}")
print(f"Amostras: {overall['total_samples']}  Accuracy: {overall['accuracy']:.4f}  "
      f"Precision: {overall['precision_macro']:.4f}  Recall: {overall['recall_macro']:.4f}  "
      f"F1: {overall['f1score_macro']:.4f}  Tempo médio/predição: {overall['avg_time_per_prediction']:.2f}s")
st = summary["prefix_length_stats"]
print(f"Tamanho dos prefixos -> min {st['min']}, max {st['max']}, média {st['average']:.2f}")

rows = [{"bucket": b, **{k: v for k, v in summary["earlyness_metrics"][b].items() if k != "considered_classes"}}
        for b in oracle._get_bucket_order() if b in summary["earlyness_metrics"]]
display(pd.DataFrame(rows).set_index("bucket").round(4))

In [ ]:
# Predições individuais (útil para inspecionar erros)
df_pred = pd.DataFrame([{
    "prefix_length": r["prefix_length"],
    "bucket": r["earlyness_bucket"],
    "expected": r["expected_answer"],
    "predicted": pred,
    "correct": r["verification_result"],
    "raw_answer": r["model_answer"][:200],
} for r, pred in zip(oracle.results, oracle.predicted_next_activities)])
display(df_pred.head(20))
df_pred.to_csv(result_file.with_suffix(".csv"), index=False)
print("CSV das predições:", result_file.with_suffix(".csv"))

In [ ]:
# (Opcional) liberar a GPU
import gc
del pipeline_instance
gc.collect(); torch.cuda.empty_cache()